# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marrwan1/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Section 1: Two paper findings + my methodology questions

Finding 1:
The paper reports that the learned model achieves Precision@50 ≈ 0.74,
a ~3x lift over the hand-written baseline rule (0.24).

Methodology question:
Where does the label come from, and is it measured on held-out data?
If the label is derived from the same window used to build features,
or if the evaluation is on training data, the lift may be overstated.
A client-holdout or time-aware split would make this claim more trustworthy.

Finding 2:
The paper identifies average ranking position as the strongest feature
in the permutation-importance analysis.

Methodology question:
Permutation importance measured on training data inflates the score of
features the model memorized. Was permutation importance computed on a
held-out test set? If not, position's reported importance may reflect
overfitting rather than a generalizable signal.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)
Split design comparison:
- Before (W05): random-style time split — train on 2026-03, test on 2026-04
- After  (W06): grouped by client — no client appears in both train and test

In [1]:
import duckdb
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

token = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

def get_features(month):
    return con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions)  AS gsc_impressions,
        SUM(gsc_clicks)       AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        CASE WHEN SUM(gsc_impressions) = 0 THEN NULL
             ELSE SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) END AS ctr,
        COUNT(DISTINCT report_date) AS days_with_data,
        SUM(gsc_clicks) AS total_clicks
    FROM read_parquet('{BASE}/month={month}/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
    """).df()

def get_label(month):
    return con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS total_clicks_next
    FROM read_parquet('{BASE}/month={month}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    """).df()

# Build full dataset
feat  = get_features("2026-03")
label = get_label("2026-04")
df = feat.merge(label, on=["client_hash_id","content_hash_id"], how="inner").dropna()
df["label"] = (df["total_clicks_next"] > df["total_clicks"] * 1.20).astype(int)

FEATURES = ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr", "days_with_data"]

# ── BEFORE: time-aware split (W05) ──────────────────────────────────────────
feat_train  = get_features("2026-03")
label_train = get_label("2026-04")
train = feat_train.merge(label_train, on=["client_hash_id","content_hash_id"], how="inner").dropna()
train["label"] = (train["total_clicks_next"] > train["total_clicks"] * 1.20).astype(int)

feat_test  = get_features("2026-04")
label_test = get_label("2026-05")
test = feat_test.merge(label_test, on=["client_hash_id","content_hash_id"], how="inner").dropna()
test["label"] = (test["total_clicks_next"] > test["total_clicks"] * 1.20).astype(int)

rf_before = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_before.fit(train[FEATURES], train["label"])
auc_before = roc_auc_score(test["label"], rf_before.predict_proba(test[FEATURES])[:, 1])

# ── AFTER: client-grouped split ──────────────────────────────────────────────
clients = df["client_hash_id"].unique()
n = len(clients)
train_clients = set(clients[:int(n * 0.8)])

train_g = df[df["client_hash_id"].isin(train_clients)]
test_g  = df[~df["client_hash_id"].isin(train_clients)]

rf_after = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_after.fit(train_g[FEATURES], train_g["label"])
auc_after = roc_auc_score(test_g["label"], rf_after.predict_proba(test_g[FEATURES])[:, 1])

print("Before/After Honest Split")
print("=" * 35)
print(f"W05 time-aware split  AUC: {auc_before:.4f}")
print(f"W06 client-grouped    AUC: {auc_after:.4f}")
print(f"Difference            : {auc_after - auc_before:+.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Before/After Honest Split
W05 time-aware split  AUC: 0.7701
W06 client-grouped    AUC: 0.8216
Difference            : +0.0515


## 3. Leakage audit

Leakage audit:

Features checked:
- gsc_impressions  : past month total — no leakage
- gsc_clicks       : past month total — no leakage
- gsc_avg_position : past month average — no leakage
- ctr              : derived from past clicks/impressions — no leakage
- days_with_data   : count of past days — no leakage

Label:
- clicks grew ≥20% in the NEXT month — future data, never used as a feature ✅

Conclusion: no leakage detected.

In [2]:
# Confirm no feature correlates suspiciously with label
corr = df[FEATURES + ["label"]].corr()["label"].drop("label").round(4)
print("Feature-label correlations (should be moderate, not perfect):")
print(corr.sort_values(ascending=False))

Feature-label correlations (should be moderate, not perfect):
days_with_data      0.1004
gsc_impressions     0.0505
gsc_clicks          0.0280
ctr                -0.0162
gsc_avg_position   -0.0942
Name: label, dtype: float64


## 4. Claim rewrite

Claim rewrite — safe language:

Before: "The model predicts which pages will grow."
After : "The model assigns a higher score to pages that,
         in the measured period, were more likely to show
         click growth — this is a directional signal, not
         a guarantee."

Before: "Random Forest outperforms the baseline."
After : "Under the client-grouped split, the Random Forest
         achieved a higher ROC-AUC than the hand-written
         rule (0.82 vs 0.64) on the observed test set.
         This is decision-support, not a production claim."

Before: "CTR and position are weak features."
After : "In this dataset and split, permutation importance
         measured on the test set showed gsc_impressions
         and days_with_data as stronger signals than CTR
         or position. This may not generalise to all clients."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.